# TMDB Movie Data Analysis

## Project Overview
This project analyzes movie data fetched from the TMDb API. It includes data cleaning, KPI calculation, advanced filtering (Directors/Cast), and visualization as per the project requirements.

In [37]:
# Importing required libraries to complete the task 
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Fetching
Fetching data from TMDb API using the provided movie IDs.

In [38]:
# Input variables to function fetch movie data
API_KEY = os.getenv('api_key')
BASE_URL = "https://api.themoviedb.org/3/movie/"

movie_ids = [
    0, 299534, 19995, 140607, 299536, 597, 135397, 420818, 24428, 
    168259, 99861, 2840, 12445, 181808, 330457, 351286, 109445, 
    321612, 260513
]

In [39]:
# Function to fetch movie data from TMDb API
def fetch_movie_data(movie_ids, api_key):
    movies_data = []
    if not api_key:
        raise ValueError("API key is required.")
    print(f"TBMb API key verified and we going to fetch data for {len(movie_ids)} movies.")
    print("Fetching data...")
    for movie_id in sorted(movie_ids):
        # Includes 'credits' to get Cast and Crew info
        url = f"{BASE_URL}{movie_id}?api_key={api_key}&language=en-US&append_to_response=credits"
        try:
            response = requests.get(url)
            if response.status_code == 200:
                movies_data.append(response.json())
            else:
                print(f"Error fetching ID {movie_id}: {response.status_code}")
        except Exception as e:
            print(f"Exception for ID {movie_id}: {e}")
    print(f"Fetched data for {len(movies_data)} movies, and failed to fetch data for {len(movie_ids) - len(movies_data)} movies.")
    return movies_data

# fetch data
raw_data = fetch_movie_data(movie_ids, API_KEY)
df = pd.DataFrame(raw_data)

TBMb API key verified and we going to fetch data for 19 movies.
Fetching data...
Error fetching ID 0: 404
Fetched data for 18 movies, and failed to fetch data for 1 movies.


## 2. Data Cleaning and Preprocessing

### Data preparation and cleaning

1. Drop irrelevant columns

In [40]:
cols_to_drop = ['adult', 'imdb_id', 'original_title', 'video', 'homepage']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')

2. Evaluating json columns

In [41]:
json_cols = ['belongs_to_collection', 'genres', 'production_companies', 
             'production_countries', 'spoken_languages']
df[json_cols].head()

,belongs_to_collection,genres,production_companies,production_countries,spoken_languages
0,None,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...","[{'id': 4, 'logo_path': '/jay6WcMgagAklUt7i9Eu...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'english_name': 'English', 'iso_639_1': 'en'..."
1,None,"[{'id': 80, 'name': 'Crime'}, {'id': 10770, 'n...","[{'id': 1245, 'logo_path': '/suKdkPTtHn0DzGYmr...","[{'iso_3166_1': 'FR', 'name': 'France'}, {'iso...","[{'english_name': 'French', 'iso_639_1': 'fr',..."
2,"{'id': 1241, 'name': 'Harry Potter Collection'...","[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'id': 174, 'logo_path': '/ingPVoHnINIrFR7WHm...","[{'iso_3166_1': 'GB', 'name': 'United Kingdom'...","[{'english_name': 'English', 'iso_639_1': 'en'..."
3,"{'id': 87096, 'name': 'Avatar Collection', 'po...","[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'id': 444, 'logo_path': None, 'name': 'Dune ...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'english_name': 'English', 'iso_639_1': 'en'..."
4,"{'id': 86311, 'name': 'The Avengers Collection...","[{'id': 878, 'name': 'Science Fiction'}, {'id'...","[{'id': 420, 'logo_path': '/hUzeosd33nzE5MCNsZ...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'english_name': 'English', 'iso_639_1': 'en'..."


3. Extract key colomn data 

In [42]:
# Extract collection names from 'belongs_to_collection' column
if 'belongs_to_collection' in df.columns:
    df['collection_name'] = df['belongs_to_collection'].apply(
        lambda x: x['name'] if isinstance(x, dict) else None
    )

In [43]:
# separate genre names by | from genres column
# separate spoken language names by | from spoken_languages column
# separate production company names by | from production_companies column
# separate production country names by | from production_countries column
def extract_names(data, key):
    if isinstance(data, list):
        return '|'.join([item['name'] for item in data if 'name' in item])
    return None
df['genre_names'] = df['genres'].apply(lambda x: extract_names(x, 'name'))
df['spoken_language'] = df['spoken_languages'].apply(lambda x: extract_names(x, 'name'))
df['production_company'] = df['production_companies'].apply(lambda x: extract_names(x, 'name'))
df['production_country'] = df['production_countries'].apply(lambda x: extract_names(x, 'name'))

4. Inspect extracted columns using value_counts() to identify anomalies.

In [44]:
extracted_cols = ['collection_name', 'genre_names', 'spoken_language', 
                  'production_company', 'production_country']

for col in extracted_cols:
    if col in df.columns:
        print(f"\n{col} values count:")
        print(df[col].value_counts())


collection_name values count:
collection_name
The Avengers Collection                4
Jurassic Park Collection               2
Frozen Collection                      2
Star Wars Collection                   2
Harry Potter Collection                1
Avatar Collection                      1
The Fast and the Furious Collection    1
The Incredibles Collection             1
The Lion King (Reboot) Collection      1
Name: count, dtype: int64

genre_names values count:
genre_names
Adventure|Action|Science Fiction             3
Action|Adventure|Science Fiction|Thriller    2
Drama|Romance                                1
Action|Adventure|Fantasy|Science Fiction     1
Science Fiction|Action|Adventure             1
Crime|TV Movie|Drama|Mystery                 1
Adventure|Fantasy                            1
Animation|Family|Adventure|Fantasy           1
Action|Adventure|Science Fiction             1
Action|Crime|Thriller                        1
Action|Adventure|Animation|Family            1
Ad

### Handling Missing & Incorrect Data

5. Convert column datatype and replace unrealistic values

In [45]:
# 5. Numeric Conversion & Handling 0/NaN
df['budget_musd'] = pd.to_numeric(df['budget'], errors='coerce') / 1000000  # Convert to Millions
df['revenue_musd'] = pd.to_numeric(df['revenue'], errors='coerce') / 1000000 # Convert to Millions
df['runtime'] = pd.to_numeric(df['runtime'], errors='coerce')
df['vote_count'] = pd.to_numeric(df['vote_count'], errors='coerce').fillna(0)

# Replace 0 with NaN for analysis where 0 is invalid
df.loc[df['budget_musd'] <= 0, 'budget_musd'] = np.nan
df.loc[df['revenue_musd'] <= 0, 'revenue_musd'] = np.nan
df.loc[df['runtime'] <= 0, 'runtime'] = np.nan

# Date Handling
df['release_date'] = pd.to_datetime(df['release_date'])
df['release_year'] = df['release_date'].dt.year

7. Remove duplicates and drop rows with unknown 'id' or 'title'.

In [46]:
# Remove duplicates based on a hashable key (use 'id' which is hashable)
df.drop_duplicates(subset='id', inplace=True)

# remove rows with unknown id or title (require both present), how many rows and columns 
# left and cols with missing values
df = df[df['id'].notna() & df['title'].notna()]

# show shape and columns that have missing values
print(f"column: {df.shape[1]}, rows: {df.shape[0]}")
print("\nColumns with missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

column: 30, rows: 18

Columns with missing values:
backdrop_path            1
belongs_to_collection    3
collection_name          3
budget_musd              1
revenue_musd             1
dtype: int64


8. Keep only rows where at least 10 columns have non-NaN values.

In [47]:
df = df.dropna(thresh=10)

9. Filter to include only 'Released' movies, then drop 'status'.

In [48]:
if 'status' in df.columns:
    df = df[df['status'] == 'Released'].drop(columns=['status'])
 

10. Extract directors and and cast

In [49]:
# 4. Extract Director and Cast
if 'credits' in df.columns:
    def get_director(credits_json):
        if isinstance(credits_json, dict) and 'crew' in credits_json:
            directors = [m['name'] for m in credits_json['crew'] if m.get('job') == 'Director']
            return '|'.join(directors)
        return ''
    def get_cast(credits_json):
        if isinstance(credits_json, dict) and 'cast' in credits_json:
            cast = [m['name'] for m in credits_json['cast']]
            return '|'.join(cast)
        return ''
    df['director'] = df['credits'].apply(get_director)
    df['cast'] = df['credits'].apply(get_cast)

11. Add cast_size and crew size

In [50]:
# cast_size and crew_size
if 'credits' in df.columns:
    def get_cast_size(credits_json):
        if isinstance(credits_json, dict) and 'cast' in credits_json:
            return len(credits_json['cast'])
        return 0
    def get_crew_size(credits_json):
        if isinstance(credits_json, dict) and 'crew' in credits_json:
            return len(credits_json['crew'])
        return 0
    df['cast_size'] = df['credits'].apply(get_cast_size)
    df['crew_size'] = df['credits'].apply(get_crew_size)

12. Reorder columns for better readability

In [ ]:
# check if columns exist in df and sort 
sorted_cols = ['id', 'title', 'tagline', 'release_date', 'genres', 'belongs_to_collection', 
               'original_language', 'budget_musd', 'revenue_musd', 'production_companies', 
               'production_countries', 'vote_count', 'vote_average', 'popularity', 'runtime', 
               'overview', 'spoken_languages', 'poster_path', 'cast', 'cast_size', 'director', 
               'crew_size']

# check missing col
missing_cols = [col for col in sorted_cols if col not in df.columns]
print("Missing columns in cleaned DataFrame:")
print(missing_cols)

print(len(sorted_cols))
df_cleaned = df[[col for col in sorted_cols if col in df.columns]]
df_cleaned.shape

Missing columns in cleaned DataFrame:
['budget_musd', 'revenue_musd', 'cast']
22


(18, 22)

13. Remain columns profit and roi

In [ ]:
df_cleaned['profit'] = df_cleaned['revenue_musd'] - df_cleaned['budget_musd']
df_cleaned['roi'] = df_cleaned['profit'] / df_cleaned['budget_musd']

In [57]:
# save cleaned data to csv
df_cleaned.to_csv('tmdb_movies_cleaned_data.csv', index=False)

## 3. KPI Implementation

### 3.1 Rankings and Performance Analysis
Including Revenue, Budget, Profit, ROI, Rating, and Popularity.

In [63]:
def analyze_kpi(df):
    # Helper to print tables
    def print_top_bottom(metric, label, ascending=False, n=5):
        print(f"\nTop {n} {label}" if not ascending else f"\nLowest {n} {label}")
        if ascending:
            print(df.nsmallest(n, metric)[['title', metric]])
        else:
            print(df.nlargest(n, metric)[['title', metric]])

    # 1. Revenue
    print_top_bottom('revenue_musd', 'Revenue')
    
    # 2. Budget
    print_top_bottom('budget_musd', 'Budget')
    print_top_bottom('budget_musd', 'Budget', ascending=True) # Lowest Budget

    # 3. Profit
    print_top_bottom('profit_musd', 'Profit')
    print_top_bottom('profit_musd', 'Profit (Worst)', ascending=True)
    
    # 4. ROI (Budget >= 10M for meaningful comparison)
    roi_df = df[df['budget_musd'] >= 10]
    print("\nTop 5 ROI (Budget >= 10M)")
    print(roi_df.nlargest(5, 'roi')[['title', 'budget_musd', 'revenue_musd', 'roi']])
    
    # 5. Rating (Vote Count >= 10)
    rated_df = df[df['vote_count'] >= 10]
    print("\nTop 5 Rated (Votes >= 10)")
    print(rated_df.nlargest(5, 'vote_average')[['title', 'vote_average', 'vote_count']])
    
    print("\nLowest 5 Rated (Votes >= 10)")
    print(rated_df.nsmallest(5, 'vote_average')[['title', 'vote_average', 'vote_count']])
    
    # 6. Popularity
    print_top_bottom('popularity', 'Popularity')
    
    # 7 Most Voted Movies
    print_top_bottom('vote_count', 'Vote Count')

analyze_kpi(df_cleaned)


Top 5 Revenue
                           title  revenue_musd
3                         Avatar   2923.706026
12             Avengers: Endgame   2799.439100
0                        Titanic   2264.162353
8   Star Wars: The Force Awakens   2068.223624
13        Avengers: Infinity War   2052.415039

Top 5 Budget
                           title  budget_musd
12             Avengers: Endgame        356.0
10      Star Wars: The Last Jedi        300.0
13        Avengers: Infinity War        300.0
17                 The Lion King        260.0
8   Star Wars: The Force Awakens        245.0

Lowest 5 Budget
                                           title  budget_musd
2   Harry Potter and the Deathly Hallows: Part 2        125.0
6                                         Frozen        150.0
7                                 Jurassic World        150.0
15                                     Frozen II        150.0
14                          Beauty and the Beast        160.0

Top 5 Profit
          

### 3.2 Specific Search Queries

### 3.3 Comparative Analysis (Franchise & Director)

## 4. Visualizations